In [2]:
import numpy as np
import cupy as cp
import cudf
import cugraph
from cugraph.community import leiden
from cuml import UMAP
from sklearn.base import BaseEstimator
from bertopic import BERTopic
import pandas as pd
from nltk.corpus import stopwords
from cuml.feature_extraction.text import CountVectorizer as CUMLCountVectorizer
from bertopic.dimensionality import BaseDimensionalityReduction
from scipy.sparse import csr_matrix as scipy_csr
from sentence_transformers import SentenceTransformer
import re
from cuml.neighbors import NearestNeighbors          # <-- GPU NN
from cuml.decomposition import PCA as cuPCA          # optional fast 2‑D proj

In [19]:
#-----------------------------------------
# Custom Functions Needed for the Pipeline
#-----------------------------------------

def umap_leiden(
    embeddings: np.ndarray,
    n_components: int = 10,
    n_neighbors: int = 50,
    min_dist: float = 0.0,
    metric: str = "cosine",
    min_weight: float = 1e-8,
    resolution: float = 0.5,
    random_state: int = 42,
) -> np.ndarray:
    """
    Returns a NumPy array of Leiden community IDs, one per row in `embeddings`.
    """
    # ---- UMAP -------------------------------------------------
    umap_model = UMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric=metric,
        verbose=True,
        random_state=random_state,
    )
    X = umap_model.fit_transform(embeddings)          # we only need the graph
    graph = umap_model.graph_.tocoo()                # scipy COO

    print("UMAP complete, computing Leiden clustering")

    # ---- prune tiny fuzzy‑union edges on CPU ------------------
    mask = graph.data >= min_weight
    row_cpu = graph.row[mask].astype(np.int32)
    col_cpu = graph.col[mask].astype(np.int32)
    data_cpu = graph.data[mask].astype(np.float32)

    # ---- move to GPU -----------------------------------------
    row = cp.asarray(row_cpu)
    col = cp.asarray(col_cpu)
    data = cp.asarray(data_cpu)

    edges = cudf.DataFrame({"src": row, "dst": col, "weight": data})

    # ---- build cuGraph ----------------------------------------
    G = cugraph.Graph(directed=False)
    G.from_cudf_edgelist(edges, source="src", destination="dst", edge_attr="weight")

    # ---- Leiden -----------------------------------------------
    parts, _ = leiden(G, resolution=resolution)
    parts = parts.sort_values("vertex")               # ensure 0‑N order

    # ---- bring labels back to CPU ------------------------------
    labels_gpu = parts["partition"].values
    labels = cp.asnumpy(labels_gpu)                   # shape (N,)

    return X, labels

# this wrapper allows us to pass this over to BERTopic

class LeidenCluster(BaseEstimator):
    """
    Scikit‑learn compatible estimator that simply returns the
    pre‑computed Leiden labels.
    """
    def __init__(self, labels: np.ndarray):
        self.labels_ = np.asarray(labels, dtype=int)

    def fit(self, X, y=None):
        # No fitting needed – labels are already computed.
        return self

    def fit_predict(self, X, y=None):
        if len(X) != len(self.labels_):
            raise ValueError(
                f"Leiden labels length ({len(self.labels_)}) does not match X ({len(X)})"
            )
        return self.labels_

# this wrapper provides GPU-accelerated vectorization in the topic representation step

class CuMLCountVectorizerWrapper:
    def __init__(self, **kwargs):
        self.vectorizer = CUMLCountVectorizer(**kwargs)

    def fit(self, docs):
        docs_series = cudf.Series(docs)
        self.vectorizer.fit(docs_series)
        return self

    def transform(self, docs):
        docs_series = cudf.Series(docs)
        X = self.vectorizer.transform(docs_series)
        # Convert CuPy CSR -> SciPy CSR for BERTopic compatibility
        return scipy_csr(X.get())

    def fit_transform(self, docs):
        docs_series = cudf.Series(docs)
        X = self.vectorizer.fit_transform(docs_series)
        return scipy_csr(X.get())

    def get_feature_names_out(self):
        vocab = self.vectorizer.get_feature_names()
        vocab_cpu = vocab.to_pandas()
        return np.array(vocab_cpu, dtype=object)

    # Required for BERTopic to access vectorizer parameters
    def get_params(self, deep=True):
        return self.kwargs

# we need to pass this empty dimensionality reduction model to BERTopic (built-in to BERTopic)

empty_dimensionality_model = BaseDimensionalityReduction()

# preprocess dataset

regex_pattern = re.compile(r'\b(?:https?://|www\.|[a-z0-9.-]+\.[a-z]{2,})[^\s]*\b|@[aA-zZ]+\b')

def clean_message_text(message):
    cleaned = message.replace("\n", " ")
    cleaned = re.sub(regex_pattern, "", cleaned)
    return cleaned

def preprocess_data(datapath, regex_pattern):
    df = pd.read_parquet(datapath)
    df = df[df["message_text"].str.len() > 2] # we will do this again, but we filter out the short texts before doing the regex call
    df["message_text"] = df["message_text"].apply(lambda x: clean_message_text(x)) # fiter out URLs and @mentions
    df = df[df["message_text"].str.len() > 2] # filter out short texts again
    outfile = datapath.split(".pqt")[0] + "_cleaned_preprocessed.pqt"
    df.to_parquet(outfile)

dm_sw = stopwords.words("dutch")
en_sw = stopwords.words("english")
all_sw = dm_sw + en_sw

vectorizer_model = CuMLCountVectorizerWrapper(
    stop_words=all_sw,
    min_df=5,
    ngram_range=(1,1)
)

In [ ]:
#-----------------------------------------
# Step 0 - Preprocess Data
#-----------------------------------------

preprocess_data("nl_all.pqt", regex_pattern)

In [20]:
# ----------------------------------------
# Step 1 - Initialize with Data 
# and Sentence Transformer Model
# ----------------------------------------
parquet_file = "nl_clean_preprocessed.pqt"

embedding_model_name = "GroNLP/bert-base-dutch-cased"
device = "cuda"

batch_size_embed = 256

# load parquet

print("Loading data")
df = pd.read_parquet(parquet_file)
print("Dataset shape: ", df.shape)

texts = df["message_text"].tolist()
record_ids = df["record_id"].tolist()

Loading data
Dataset shape:  (1468488, 16)


In [ ]:
#-----------------------------------------
# Step 2 - Sentence Embeddings
#-----------------------------------------

# Load model
sentence_model = SentenceTransformer(embedding_model_name)
"""
# Force truncation length (<= model’s max_position_embeddings)
print("HF max_position_embeddings:",
      sentence_model[0].auto_model.config.max_position_embeddings)  # typically 514
sentence_model.max_seq_length = 256  # this is what encode() will truncate to
"""

# Encode with truncation handled internally
sentence_embeddings = sentence_model.encode(
    texts,
    batch_size=batch_size_embed,
    show_progress_bar=True,
)

np.save("es_full_sample.npy", sentence_embeddings)

In [21]:
sentence_embeddings = np.load("nl_bertje_full_sample.npy")

In [22]:
X, leiden_cluster_labels = umap_leiden(
    embeddings = sentence_embeddings,
    n_components = 2,
    n_neighbors = 50,
    min_dist = 0.0,
    min_weight = 1e-3,
    resolution=1.0,
    random_state = 42
)

print(pd.Series(leiden_cluster_labels).value_counts())

[2026-05-21 16:01:59.262] [CUML] [debug] Building knn graph using build_algo='brute_force_knn'
[2026-05-21 16:01:59.622] [CUML] [debug] Computing KNN Graph
[2026-05-21 16:05:41.866] [CUML] [debug] Computing fuzzy simplicial set
UMAP complete, computing Leiden clustering
0     128067
1     111565
4     100109
20     74150
10     70365
       ...  
96         7
97         7
85         7
89         7
82         7
Name: count, Length: 98, dtype: int64


In [ ]:
pd.Series(leiden_cluster_labels).value_counts()

In [23]:
cluster_model = LeidenCluster(leiden_cluster_labels)

In [24]:
class CuMLCountVectorizerWrapper:
    """
    GPU‑accelerated CountVectorizer that returns a SciPy CSR matrix
    (compatible with BERTopic).  The only change w.r.t. the original
    wrapper is the explicit cast of the categorical column to uint32
    before renaming the categories – this prevents the
    “dtype category does not match … UINT32” error.
    """
    def __init__(self, **kwargs):
        self.kwargs = kwargs
        self.vectorizer = CUMLCountVectorizer(**kwargs)

    # -----------------------------------------------------------------
    # Helper that forces the categorical column to uint32 before
    # renaming categories – required for CuML ≥ 24.04
    # -----------------------------------------------------------------
    @staticmethod
    def _fix_category_dtype(df, col_name, new_categories):
        """
        df          : cudf.DataFrame containing a categorical column
        col_name    : name of the column to fix
        new_categories : 1‑D uint32 array (the indices we want to keep)
        """
        # Cast the column to uint32 codes (the underlying storage type)
        df[col_name] = df[col_name].astype("category")
        # `set_categories` expects the new categories to have the same dtype
        # as the column’s codes.  We therefore pass a uint32 Series.
        df[col_name] = df[col_name].cat.set_categories(
            cudf.Series(new_categories, dtype="uint32")
        ).cat.codes
        return df

    # -----------------------------------------------------------------
    def fit(self, docs):
        docs_series = cudf.Series(docs)
        self.vectorizer.fit(docs_series)
        return self

    def transform(self, docs):
        docs_series = cudf.Series(docs)
        X = self.vectorizer.transform(docs_series)

        # ---- Fix the dtype mismatch (only needed when pruning features) ----
        if hasattr(self.vectorizer, "_vocab"):
            # `_vocab` is a cudf.Series of token strings; its index are uint32
            keep_idx = self.vectorizer._vocab.index.values.astype("uint32")
            # The internal dataframe used by CuML is stored in `X._data`
            # (a cuDF DataFrame with columns ["token", "count"])
            # We need to rename the categorical column "token".
            X = self._fix_category_dtype(X, "token", keep_idx)

        # Convert CuPy CSR → SciPy CSR (BERTopic expects SciPy)
        return scipy_csr(X.get())

    def fit_transform(self, docs):
        docs_series = cudf.Series(docs)
        X = self.vectorizer.fit_transform(docs_series)

        # Same dtype‑fix as in `transform`
        if hasattr(self.vectorizer, "_vocab"):
            keep_idx = self.vectorizer._vocab.index.values.astype("uint32")
            X = self._fix_category_dtype(X, "token", keep_idx)

        return scipy_csr(X.get())

    # -----------------------------------------------------------------
    def get_feature_names_out(self):
        vocab = self.vectorizer.get_feature_names()
        # `vocab` is a cudf.Series of strings → bring to host memory
        return np.array(vocab.to_pandas(), dtype=object)

    def get_params(self, deep=True):
        # BERTopic calls this to clone the vectorizer
        return self.kwargs

vectorizer_model = CuMLCountVectorizerWrapper(
    stop_words=all_sw,
    min_df=1,
    ngram_range=(1,1)
)

In [25]:
topic_model = BERTopic(
    embedding_model = None,
    umap_model = empty_dimensionality_model,
    hdbscan_model = cluster_model,
    vectorizer_model=vectorizer_model,
    verbose=True
)

In [26]:
topics, probs = topic_model.fit_transform(texts, embeddings=sentence_embeddings)

2026-05-21 16:06:34,707 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-21 16:06:34,708 - BERTopic - Dimensionality - Completed ✓
2026-05-21 16:06:37,623 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-21 16:06:37,687 - BERTopic - Cluster - Completed ✓
2026-05-21 16:06:37,808 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-21 16:09:15,353 - BERTopic - Representation - Completed ✓


In [27]:
topic_model.save("nl_50_nn_bertje_leiden_clustered.bertopic", serialization="pickle")

2026-05-21 16:09:21,422 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [28]:
topic_model.get_topic_info().to_html("nl_50_nn_bertje_leiden_clustered_topic_info.html")
intertopic_distance_viz = topic_model.visualize_topics()
intertopic_distance_viz.write_html("nl_50_nn_bertje_leiden_clustered_intertopic_viz.html")
df["topic_label_leiden"] = pd.Series(topics)
df.to_parquet("nl_50_nn_bertje_leiden_labeled.pqt")